# 🧪 W3-D6 概念实验：SFT 全流程模拟

> 配套阅读：`ima/第3周-Day6-SFT全流程实战.md`
>
> 用纯 Python 模拟 SFT 每个环节：数据构造、分词、LoRA、微调前后对比。

## 实验 1：SFT 数据构造

SFT 数据 = (instruction, output) 对。

In [ ]:
sft_data = [
    {"instruction": "糖水店的招牌产品是什么？", "output": "我们的招牌是手工现熬红豆沙和绿豆沙，选用优质原料慢火熬制4小时。"},
    {"instruction": "你们几点开门？", "output": "每天上午10:00到晚上22:00，全年无休。"},
    {"instruction": "红豆沙多少钱？", "output": "小碗8元，大碗12元。红豆沙+杨枝甘露套餐25元。"},
    {"instruction": "夏天有什么推荐？", "output": "夏天推荐：杨枝甘露18元、绿豆沙8元、椰汁西米露12元。"},
    {"instruction": "你们支持外卖吗？", "output": "支持美团和饿了么，3公里内满30元免配送费。"},
    {"instruction": "有过敏原吗？", "output": "部分产品含牛奶、花生、芒果。如有过敏史请提前告知。"},
]
print(f"SFT 数据集：{len(sft_data)} 条指令-回答对")
print(f"示例：{sft_data[0]['instruction']} → {sft_data[0]['output'][:30]}…")

## 实验 2：模拟分词与训练 Loss

字符级分词 → 交叉熵 Loss。

In [ ]:
import numpy as np
from collections import Counter

vocab = {}; idx = 0
for d in sft_data:
    for ch in list(d['instruction']+d['output']):
        if ch not in vocab: vocab[ch] = idx; idx += 1
print(f"词表大小: {len(vocab)} 个字符")

char_freq = Counter()
for d in sft_data: char_freq.update(d['output'])
total_c = sum(char_freq.values())
char_prob = {ch: cnt/total_c for ch, cnt in char_freq.items()}

losses = []
for d in sft_data:
    toks = list(d['output'])
    nll = -sum(np.log(char_prob.get(t, 1e-10)) for t in toks) / len(toks)
    losses.append(nll)

print("各样本 Cross-Entropy Loss:")
for i, l in enumerate(losses): print(f"  样本{i+1}: {l:.3f}")
print(f"平均: {np.mean(losses):.3f}")

## 实验 3：模拟 LoRA 参数更新

$W_{new} = W_{orig} + B \times A$，只训练 B 和 A。

In [ ]:
import numpy as np

d, r = 8, 2
np.random.seed(8)
W = np.random.randn(d, d) * 0.5
A = np.random.randn(r, d) * 0.01
B = np.random.randn(d, r) * 0.01
print(f"原始权重: {d}×{d}={d*d}（冻结）| LoRA: {2*d*r} 可训练，占比 {2*d*r/(d*d)*100:.1f}%")
for step in range(3):
    B -= 0.01 * np.random.randn(d, r) * 0.1
    A -= 0.01 * np.random.randn(r, d) * 0.1
    pct = np.abs(B@A).sum() / np.abs(W).sum() * 100
    print(f"Step {step+1}: LoRA 修改量占总权重 {pct:.2f}%")

## 实验 4：微调前后回答质量对比

从「只会续写」到「学会回答指令」。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import font_manager
font_path = "/usr/share/fonts/opentype/noto/NotoSansCJK-Regular.ttc"
font_manager.fontManager.addfont(font_path)
font_name = font_manager.FontProperties(fname=font_path).get_name()
plt.rcParams["font.family"] = font_name
plt.rcParams["axes.unicode_minus"] = False
questions = ['招牌产品','营业时间','价格查询','外卖服务','过敏原','夏日推荐']
pre, post = [15,8,20,5,10,12], [85,92,88,90,82,86]
x = np.arange(len(questions)); w = 0.35
fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(x-w/2, pre, w, label='SFT 前（只会续写）', color='#e74c3c', alpha=0.8)
ax.bar(x+w/2, post, w, label='SFT 后（学会回答）', color='#3498db', alpha=0.8)
ax.set_xticks(x); ax.set_xticklabels(questions, fontsize=10)
ax.set_ylabel('回答质量分数')
ax.set_title('SFT 微调前后：从「续写」到「回答」')
ax.legend(fontsize=11); ax.grid(True, alpha=0.3, axis='y'); plt.tight_layout(); plt.show()
print(f"SFT 前 {np.mean(pre):.1f} → 后 {np.mean(post):.1f} (+{np.mean(post)-np.mean(pre):.1f})")

## 结论

| 问题 | 实验证据 |
|---|---|
| SFT 数据 | 实验1：指令-回答对 |
| 训练目标 | 实验2：最大化回答概率 |
| LoRA 原理 | 实验3：冻结原始+小补丁 |
| 前后差异 | 实验4：从续写到回答 |

→ 配套阅读：`ima/第3周-Day6-SFT全流程实战.md`